In [2]:
import os
import faiss
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from groq import Groq

### Loading dataset

In [3]:
print("Loading dataset from Hugging face [wikipedia passages formatted for RAG]")
dataset = load_dataset("rag-datasets/rag-mini-wikipedia","text-corpus")
print(dataset)

Loading dataset from Hugging face [wikipedia passages formatted for RAG]


DatasetDict({
    passages: Dataset({
        features: ['passage', 'id'],
        num_rows: 3200
    })
})


In [4]:
first_row = dataset['passages'][0]
print('1st row:',first_row.keys())
print('sample passages text:',first_row['passage'])

1st row: dict_keys(['passage', 'id'])
sample passages text: Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part of South America.  It is home to 3.3 million people, of which 1.7 million live in the capital Montevideo and its metropolitan area.


In [5]:
raw_passages = dataset['passages']['passage']
print(f"Loaded {len(raw_passages)} base passges from dataset")

Loaded 3200 base passges from dataset


### Inspect & chunking 

In [6]:
print("Inspect & Chunk passages")
avg_len = sum(len(p) for p in raw_passages) / len(raw_passages)
print(avg_len,"charcters in each passages")

Inspect & Chunk passages
389.848125 charcters in each passages


In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len
)
chunks = []
for t in raw_passages:
    split_text = splitter.split_text(t)
    chunks.extend(split_text)
print(f'total chunks created:{len(chunks)}')

total chunks created:4492


### Generating embedding

In [8]:
print('Generating embedding')
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device = "cpu")

# convert chunks to diemnsional vectors
embeddings = embedding_model.encode(chunks, batch_size = 32, show_progress_bar = True, convert_to_numpy = True)

# Normalize vectors(consine similarity in faiss)
faiss.normalize_L2(embeddings)
print(f'Embeddings matrix shape: {embeddings.shape}')

Generating embedding


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/141 [00:00<?, ?it/s]

Embeddings matrix shape: (4492, 384)


### Build faiss index & save to disk 

In [9]:
embedding_dim = embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings)

os.makedirs('vector_store',exist_ok = True)
faiss.write_index(index, 'vector_store/index.faiss')
np.save('vector_store/chunks.npy', np.array(chunks, dtype= object))

print(f'successfully processed & stored {len(chunks)} chunks')
print('faiss index saved at: vector_store/index.faiss')
print('text chunks saved at: vector_store/chunks.npy')

successfully processed & stored 4492 chunks
faiss index saved at: vector_store/index.faiss
text chunks saved at: vector_store/chunks.npy


### Retriving & Generating

In [10]:
# loading vector index & chunks 
index = faiss.read_index('vector_store/index.faiss')
chunks = np.load('vector_store/chunks.npy', allow_pickle = True)

embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device = "cpu")
groq_client = Groq(api_key = "YOUR_API_KEY")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
def retrieve_chunks(query : str, top_k : int = 3):
    query_vector = embedding_model.encode([query], convert_to_numpy = True)
    faiss.normalize_L2(query_vector)
    score, indices = index.search(query_vector, top_k)

    retrieved = []
    for s, i in zip(score[0], indices[0]):
        retrieved.append({
            "score":float(s),
            "text": chunks[i]
        })
    return retrieved

In [12]:
def ask_rag(question: str, top_k : int = 3):
    result = retrieve_chunks(question, top_k = top_k)
    context_text = '\n\n'.join([r['text'] for r in result])
    
    prompt = f"Answer the question using only the context below.If the answer isn't in the context, say so.\n\nContext: {context_text}\n\nQuestion: {question}"
    completion = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.2
)

    answer = completion.choices[0].message.content
    retrieved_chunks = [r['text'] for r in result]
    return answer, retrieved_chunks

## API Testing

In [13]:
import nest_asyncio
import asyncio
import uvicorn 
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

In [14]:
nest_asyncio.apply()

app = FastAPI(title = 'Wikipedia Q&A RAG System')

class QueryRequest(BaseModel):
    question : str
    top_k : int = 3
    
class QueryResponse(BaseModel):
    question : str
    answer : str
    retrieved_chunks : list[str]

@app.get("/health")
def health_check():
    return {
        "status":"ok",
        "message":"RAG API is up and running"
    }

@app.post("/query", response_model = QueryResponse)
def query_rag(request : QueryRequest):
    if not request.question.strip():
        raise HTTPException(status_code = 404, detail = 'Question cannot be empty')

    try:
        answer, chunks = ask_rag(request.question, request.top_k)
        return QueryResponse(
            question = request.question,
            answer = answer,
            retrieved_chunks = chunks
        )
    except Exception as e:
        raise HTTPException(status_code = 500, detail = str(e))


config = uvicorn.Config(app, host = "127.0.0.1", port = 8007)
server = uvicorn.Server(config)
asyncio.create_task(server.serve())

<Task pending name='Task-1' coro=<Server.serve() running at C:\Users\snaaf\AppData\Roaming\Python\Python313\site-packages\uvicorn\server.py:88>>

INFO:     Started server process [1552]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8007 (Press CTRL+C to quit)


INFO:     127.0.0.1:61894 - "GET / HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:61894 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:61894 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:61894 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:55836 - "POST /query HTTP/1.1" 200 OK
